# QuRater 4 interpretable axes

QuRater-1.3B outputs 4 logits per doc: `writing_style`, `required_expertise`, `facts_and_trivia`, `educational_value`.

Currently scored on **32/100 shards** (~520k docs). Enough for marginal + pairwise EDA; don't trust within-topic conditional stats for rare topics yet.

Goal: **decide whether QuRater adds signal beyond DCLM/FWEDU/PC filters**. If `educational_value` is r≈0.95 with FWEDU it's redundant. If each axis carries independent signal, keep all 4.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import zstandard as zstd, io, json as _json

from config import DATA_ROOT, CORPUS_ROOT, ANNOTATIONS_PARQUET, shard_stem, doc_path

sns.set_context('notebook')
plt.rcParams['figure.dpi'] = 100

AXES = ['writing_style', 'required_expertise', 'facts_and_trivia', 'educational_value']
PC_MODELS = ['arc_easy', 'piqa', 'sciq', 'lambada']
OTHER_SCORES = ['dclm_score', 'fwedu_score'] + PC_MODELS

In [ ]:
# Build dataframe: for each shard that has QuRater, load QuRater + annotations + PC scores.
qdir = DATA_ROOT / 'scores_qurater'
qfiles = sorted(qdir.glob('CC_shard_*.npy'))
q_shards = sorted(int(p.stem.split('_')[2]) for p in qfiles)
print(f'QuRater available on {len(q_shards)} shards: {q_shards[:5]}...{q_shards[-3:]}')

annot = pd.read_parquet(ANNOTATIONS_PARQUET)
annot = annot[annot.shard_idx.isin(q_shards)].sort_values(['shard_idx','doc_idx']).reset_index(drop=True)

q_arrays = [np.load(qdir / f'{shard_stem(s)}.npy') for s in q_shards]
q = np.concatenate(q_arrays, axis=0)
assert len(q) == len(annot), (len(q), len(annot))
for i, a in enumerate(AXES):
    annot[a] = q[:, i]

for m in PC_MODELS:
    chunks = []
    for s in q_shards:
        p = DATA_ROOT / 'scores_pc' / m / f'{shard_stem(s)}.npy'
        chunks.append(np.load(p) if p.exists() else np.full(int((annot.shard_idx == s).sum()), np.nan, np.float32))
    annot[m] = np.concatenate(chunks)
print(f'{len(annot):,} docs with all axes + filters')

## 1. Marginal distributions of 4 axes
These are logits (real-valued), roughly Gaussian-ish from pairwise ranking loss. Check for skew or outliers.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, a in zip(axes.flat, AXES):
    x = annot[a].dropna()
    ax.hist(x, bins=100)
    ax.axvline(x.mean(), color='k', lw=1, ls='--')
    ax.set_title(f'{a}\n mean={x.mean():.2f}, std={x.std():.2f}, skew={stats.skew(x):.2f}')
fig.tight_layout(); plt.show()

## 2. Pairwise among the 4 axes

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
sns.heatmap(annot[AXES].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Pearson correlation among QuRater axes')
plt.show()

In [ ]:
sub = annot[AXES].dropna().sample(min(50_000, len(annot)), random_state=0)
fig, axes = plt.subplots(4, 4, figsize=(13, 13))
for i, a in enumerate(AXES):
    for j, b in enumerate(AXES):
        ax = axes[i, j]
        if i == j:
            ax.hist(sub[a], bins=60); ax.set_yticks([])
        else:
            ax.hexbin(sub[b], sub[a], gridsize=30, bins='log', cmap='viridis')
        if i == 3: ax.set_xlabel(b)
        if j == 0: ax.set_ylabel(a)
fig.tight_layout(); plt.show()

## 3. QuRater axes vs DCLM / FWEDU / perplexity-correlations
Money plot for "does QuRater add signal?". In particular: does `educational_value` = FWEDU? Does `writing_style` = DCLM?

In [ ]:
full = annot[AXES + OTHER_SCORES].dropna()
cross = full.corr('spearman').loc[AXES, OTHER_SCORES]
fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(cross, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Spearman correlation: QuRater axes × other filters')
plt.show()

## 4. QuRater axes by topic / format

In [ ]:
topic_means = annot.groupby('topic')[AXES].mean()
topic_means_z = (topic_means - topic_means.mean()) / topic_means.std()
fmt_means = annot.groupby('format')[AXES].mean()
fmt_means_z = (fmt_means - fmt_means.mean()) / fmt_means.std()

fig, axes = plt.subplots(1, 2, figsize=(12, 8))
sns.heatmap(topic_means_z, annot=True, fmt='.1f', cmap='coolwarm', center=0, ax=axes[0]); axes[0].set_title('Topic mean (z)')
sns.heatmap(fmt_means_z,   annot=True, fmt='.1f', cmap='coolwarm', center=0, ax=axes[1]); axes[1].set_title('Format mean (z)')
fig.tight_layout(); plt.show()

## 5. PCA on all 10 scores
Single "general quality" PC, or distinct dimensions? Eigenvalue spectrum + loadings.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

all_cols = AXES + OTHER_SCORES
X = annot[all_cols].dropna().values
Xs = StandardScaler().fit_transform(X)
pca = PCA().fit(Xs)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(range(1, len(pca.explained_variance_ratio_)+1), pca.explained_variance_ratio_)
axes[0].set_xlabel('PC'); axes[0].set_ylabel('var explained'); axes[0].set_title('Scree')
loadings = pd.DataFrame(pca.components_[:4].T, index=all_cols, columns=[f'PC{i+1}' for i in range(4)])
sns.heatmap(loadings, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[1])
axes[1].set_title('Loadings of top 4 PCs')
fig.tight_layout(); plt.show()

## 6. Qualitative extremes (human-readable sanity check)
Top-5 and bottom-5 documents per axis with a text snippet. Most useful check for "does QuRater understand what it claims to measure": if `educational_value` top-5 look like textbooks and bottom-5 look like forum spam, it's real.

In [ ]:
def load_texts(shard_idx, doc_idxs):
    out = {}
    needed = set(int(d) for d in doc_idxs)
    with open(doc_path(shard_idx), 'rb') as f:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(f) as r:
            reader = io.TextIOWrapper(r, encoding='utf-8')
            for i, line in enumerate(reader):
                if i in needed:
                    out[i] = _json.loads(line).get('text', '')
                    if len(out) == len(needed):
                        break
    return out

def show_extremes(axis, k=5, snippet_chars=400):
    df = annot[['shard_idx', 'doc_idx', 'topic', 'format', axis]].dropna()
    top = df.nlargest(k, axis)
    bot = df.nsmallest(k, axis)
    for label, sel in [('TOP', top), ('BOT', bot)]:
        print(f'\n===== {label} {k} on {axis} =====')
        for shard_idx, grp in sel.groupby('shard_idx'):
            texts = load_texts(int(shard_idx), grp.doc_idx.values)
            for _, row in grp.iterrows():
                t = texts.get(int(row.doc_idx), '').replace('\n', ' ')
                print(f'  score={row[axis]:+.2f}  topic={row.topic} fmt={row.format}  shard={shard_idx} doc={row.doc_idx}')
                print(f'    {t[:snippet_chars]!r}')

show_extremes('educational_value', k=5)

In [ ]:
show_extremes('writing_style', k=5)

In [ ]:
show_extremes('required_expertise', k=5)

In [ ]:
show_extremes('facts_and_trivia', k=5)